# Day 15: Debugging, edit distance, sequence datasets

*Week 3: PyTorch fundamentals and dynamic programming*

**Time budget:** about 60–90 minutes. **A** is Python fluency (~15 min), **B** is an interview-style algorithm (~25 min), **C** is NumPy / PyTorch / ML (~30 min).

**For each problem:**
1. Plan on paper or in comments before you type any code.
2. Work on it for 20 minutes before asking for a hint.
3. Run the test cell under it. Don't edit the tests.
4. Answer the **Say out loud** prompts, out loud. Interviews grade the explanation as much as the code.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## A. Find the bugs

`PyTorch`

"Here's some code, what's wrong with it?" is a very common ML interview format. This loop is supposed to fit `y = 3x + 1`, but it has **three** bugs:

```python
def train_buggy(model, X, y, epochs=200, lr=0.1):
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    history = []
    for epoch in range(epochs):
        preds = model(X)            # shape (N, 1)
        loss = loss_fn(preds, y)    # y has shape (N,)
        loss.backward()
        opt.step()
        history.append(loss)
    return history
```

1. In a markdown cell, list each bug: what goes wrong and why. One of them gives **no error at all**, just a wrong answer, and PyTorch only prints a warning. Those are the dangerous ones.
2. Write `train_fixed` with the same signature, returning a list of floats.

Try to spot the bugs **before** running anything.

In [ ]:
def train_fixed(model, X, y, epochs=200, lr=0.1):
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
torch.manual_seed(0)
X = torch.rand(100, 1) * 2 - 1
y = 3 * X[:, 0] + 1
model = nn.Linear(1, 1)
hist = train_fixed(model, X, y)
assert all(isinstance(h, float) for h in hist), "history should hold floats, not tensors"
assert hist[-1] < 1e-3, f"final loss {hist[-1]:.4f}"
assert abs(model.weight.item() - 3) < 0.05 and abs(model.bias.item() - 1) < 0.05
print("✅ A passed")

## B. Edit distance

`Algorithms`

Write `edit_distance(a, b)`: the minimum number of single-character insertions, deletions or substitutions needed to turn `a` into `b` (Levenshtein distance).

```
edit_distance("horse", "ros") -> 3
```

Define `dp[i][j]` = the edit distance between the first `i` characters of `a` and the first `j` characters of `b`. Write down the base cases (`dp[i][0]` and `dp[0][j]`) and the recurrence before coding.

**Say out loud:** time and space complexity. Can you reduce the space to two rows?

In [ ]:
def edit_distance(a, b):
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
assert edit_distance("horse", "ros") == 3
assert edit_distance("intention", "execution") == 5
assert edit_distance("", "abc") == 3 and edit_distance("abc", "") == 3
assert edit_distance("same", "same") == 0
assert edit_distance("kitten", "sitting") == 3
print("✅ B passed")

## C. A sequence Dataset for the MDN-RNN

`PyTorch`

This is the Dataset you'll need for your M model.

Write `class SequenceDataset(Dataset)`. Its `__init__(self, episodes, seq_len)` takes a list of episode dicts, each with:
- `"z"`: array of shape (T+1, z_dim), the latents for all T+1 frames;
- `"actions"`: array of shape (T, a_dim).

Cut each episode into **non-overlapping** chunks of `seq_len` steps, dropping any leftover steps at the end. For a chunk starting at step `t`, `__getitem__` returns three float32 tensors:
- `z_in = z[t : t+L]`
- `a = actions[t : t+L]`
- `z_next = z[t+1 : t+L+1]`, the **target**, shifted by one step.

Order the items episode by episode, chunk by chunk.

**Hint:** in `__init__`, build a list of `(episode_index, start_step)` pairs. Then `__len__` and `__getitem__` become one-liners.

In [ ]:
from torch.utils.data import Dataset


class SequenceDataset(Dataset):
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
def make_ep(offset):
    return {"z": (np.arange(126 * 4).reshape(126, 4) + offset).astype(np.float32),
            "actions": (np.arange(125).reshape(125, 1) + offset).astype(np.float32)}
eps = [make_ep(0), make_ep(10_000)]
ds = SequenceDataset(eps, seq_len=32)
assert len(ds) == 6, f"125 // 32 = 3 chunks per episode, 2 episodes -> 6, got {len(ds)}"
z_in, a, z_next = ds[0]
assert z_in.shape == (32, 4) and a.shape == (32, 1) and z_next.shape == (32, 4)
assert z_in.dtype == torch.float32
assert torch.equal(z_in, torch.from_numpy(eps[0]["z"][0:32]))
assert torch.equal(z_next, torch.from_numpy(eps[0]["z"][1:33])), "target must be shifted by one"
assert torch.equal(z_next[:-1], z_in[1:])
z_in, a, z_next = ds[5]   # episode 1, chunk 2 -> starts at step 64
assert torch.equal(z_in, torch.from_numpy(eps[1]["z"][64:96]))
assert torch.equal(a, torch.from_numpy(eps[1]["actions"][64:96]))
from torch.utils.data import DataLoader
zb, ab, znb = next(iter(DataLoader(ds, batch_size=4)))
assert zb.shape == (4, 32, 4)
print("✅ C passed")

## Wrap-up (5 minutes)

- For each problem: what's its time and space complexity? Which edge cases did you handle?
- Anything that took more than 20 minutes, or needed a hint: add it to the **Redo list** in `README.md` and solve it again from scratch in 3 days, without looking at today's code.
- Tick off today in `README.md`.